In [3]:
pip install pycryptodome


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 28.0 MB/s eta 0:00:00


In [4]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import json
import random
import time
from datetime import datetime


# =========================
#   (1) IoT Sensor (Device)
# =========================

def generate_sensor_data():
    """Generate random temperature and humidity readings."""
    temperature = round(random.uniform(20.0, 35.0), 2)   # degrees Celsius
    humidity = round(random.uniform(30.0, 80.0), 2)      # percentage
    payload = {
        "device_id": "sensor-01",
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "temperature": temperature,
        "humidity": humidity
    }
    return payload


def encrypt_payload(key: bytes, data: dict):
    """
    Encrypt JSON payload using AES-GCM.
    Returns (nonce, ciphertext, tag).
    """
    # Convert dict to JSON bytes
    plaintext = json.dumps(data).encode("utf-8")

    # For GCM, a 12-byte nonce is typical
    nonce = get_random_bytes(12)

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)

    return nonce, ciphertext, tag


# =========================
#   (2) Base Station (Server)
# =========================

def decrypt_payload(key: bytes, nonce: bytes, ciphertext: bytes, tag: bytes):
    """
    Decrypt payload using AES-GCM and verify authenticity.
    Returns the original dict if successful.
    """
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    plaintext = cipher.decrypt_and_verify(ciphertext, tag)
    data = json.loads(plaintext.decode("utf-8"))
    return data


# =========================
#   (3) Simulation
# =========================

def simulate_secure_transmission():
    # In real IoT, the key is pre-shared between device and server.
    # 16 bytes = 128-bit AES key (lightweight enough for many IoT devices)
    key = get_random_bytes(16)

    # 1. Device generates sensor data
    sensor_data = generate_sensor_data()
    print("=== Original Sensor Data (Before Encryption) ===")
    print(json.dumps(sensor_data, indent=2))
    print()

    # 2. Device encrypts the data before sending
    nonce, ciphertext, tag = encrypt_payload(key, sensor_data)

    print("=== Encrypted Data (What goes over the air) ===")
    print(f"Nonce     (hex): {nonce.hex()}")
    print(f"Ciphertext(hex): {ciphertext.hex()}")
    print(f"Tag       (hex): {tag.hex()}")
    print()

    # 3. Simulate transmission delay / network
    time.sleep(0.5)  # just to simulate time passing

    # 4. Server receives (nonce, ciphertext, tag) and decrypts
    received_data = decrypt_payload(key, nonce, ciphertext, tag)

    print("=== Decrypted Sensor Data (At Base Station) ===")
    print(json.dumps(received_data, indent=2))


if __name__ == "__main__":
    simulate_secure_transmission()


/tmp/ipython-input-2555845216.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


=== Original Sensor Data (Before Encryption) ===
{
  "device_id": "sensor-01",
  "timestamp": "2025-11-30T11:32:37.622261Z",
  "temperature": 21.29,
  "humidity": 53.03
}

=== Encrypted Data (What goes over the air) ===
Nonce     (hex): 7fc8dae10ca76bde722c5430
Ciphertext(hex): 357b32f5e75350a1fee8d1f286592bb8e4c6dab79f75d40a28c82fa017798be5e6e4b3ca86b95a173b8e1ad35a7fd569e57ab77727c6ee7fa5bfcd905bc75ae0d1bdadfb884925929f985dd15551e40c5539d69e0b9b8f18002e162d1d3b627f40f126c54515d22f5f9fc50fd7ae29
Tag       (hex): b8444a06baa3ae49dd1d9e2c1601f0b2

=== Decrypted Sensor Data (At Base Station) ===
{
  "device_id": "sensor-01",
  "timestamp": "2025-11-30T11:32:37.622261Z",
  "temperature": 21.29,
  "humidity": 53.03
}
